# Energon ⚡ - Exploratory Data Analysis

This notebook provides interactive exploratory analysis of the 15-minute electricity load dataset (370 meters from 2011 to 2014).

### Contents:
1. **Macro Dynamics**: Aggregate Grid Load & Connection Timeline
2. **Seasonal Decompositions**: Diurnal (24h), Weekly, and Annual Patterns
3. **Micro Heterogeneity**: Consumer Scale Distributions & Load Factors across 370 meters
4. **Consumer Archetypes**: Commercial, Residential, Continuous Industrial, and Intermittent
5. **Temporal Autocorrelation**: ACF & PACF Analysis for Lag Feature Engineering

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import acf, pacf, adfuller

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Load cleaned dataset from parquet
df = pd.read_parquet('data/cleaned_electricity_load.parquet')
agg = df.sum(axis=1, min_count=1)
agg.name = 'aggregate_load'
print(f"Loaded dataset: {df.shape[0]:,} intervals across {df.shape[1]} meters.")
print(f"Date Range: {df.index.min()} to {df.index.max()}")

## 1. System Macro Dynamics (Aggregate Grid Load)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})

agg_daily = agg.resample('D').mean()
agg_daily_roll7 = agg_daily.rolling(7, center=True).mean()

ax1.plot(agg_daily.index, agg_daily.values, color='#94a3b8', alpha=0.5, label='Daily Mean Load (kW)')
ax1.plot(agg_daily_roll7.index, agg_daily_roll7.values, color='#2563eb', linewidth=2, label='7-Day Smoothed Load (kW)')
ax1.axvline(pd.to_datetime('2012-01-01'), color='#dc2626', linestyle='--', label='Stable Baseline (2012-01-01)')
ax1.axvline(pd.to_datetime('2014-01-01'), color='#16a34a', linestyle='--', label='2014 Test Split')
ax1.set_title('System-Wide Aggregate Electricity Load (2011–2014)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Average Load (kW)')
ax1.legend(loc='upper left')

active_meters = df.notna().sum(axis=1).resample('D').median()
ax2.plot(active_meters.index, active_meters.values, color='#d97706', linewidth=1.8)
ax2.set_title('Active Connected Meters Over Time', fontsize=11, fontweight='bold')
ax2.set_ylabel('Active Count')
ax2.set_ylim(0, 390)
plt.tight_layout()
plt.show()

## 2. Diurnal, Weekly, and Annual Seasonality

In [ ]:
agg_stable = agg.loc['2012-01-01':'2014-12-31']
df_feat = pd.DataFrame({'load': agg_stable})
df_feat['hour'] = df_feat.index.hour + df_feat.index.minute / 60.0
df_feat['day_of_week'] = df_feat.index.day_name()
df_feat['is_weekend'] = df_feat.index.dayofweek.isin([5, 6])
df_feat['month'] = df_feat.index.month_name()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Diurnal
wk = df_feat[~df_feat['is_weekend']].groupby('hour')['load'].mean()
we = df_feat[df_feat['is_weekend']].groupby('hour')['load'].mean()
axes[0].plot(wk.index, wk.values, color='#2563eb', linewidth=2.5, label='Weekday')
axes[0].plot(we.index, we.values, color='#ea580c', linewidth=2.5, label='Weekend')
axes[0].set_title('24-Hour Diurnal Demand Profile', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Aggregate Load (kW)')
axes[0].set_xticks(range(0, 25, 4))
axes[0].legend()

# Weekly
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.boxplot(data=df_feat, x='day_of_week', y='load', order=day_order, ax=axes[1], hue='day_of_week', palette='Blues_r', legend=False, showfliers=False)
axes[1].set_title('Day-of-Week Variation', fontweight='bold')
axes[1].tick_params(axis='x', rotation=35)

# Monthly
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
sns.boxplot(data=df_feat, x='month', y='load', order=month_order, ax=axes[2], hue='month', palette='Spectral_r', legend=False, showfliers=False)
axes[2].set_title('Annual Monthly Seasonality', fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Micro Consumer Heterogeneity (370 Meters)

In [ ]:
df_stable = df.loc['2012-01-01':'2014-12-31']
mean_load = df_stable.mean()
max_load = df_stable.max()
load_factor = mean_load / max_load
corrs = df_stable.corrwith(agg_stable)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.histplot(mean_load, bins=40, kde=True, ax=axes[0], color='#0284c7', log_scale=True)
axes[0].set_title('Consumer Scale Distribution (kW)', fontweight='bold')
axes[0].set_xlabel('Mean Load (kW) [Log Scale]')

sns.histplot(load_factor, bins=35, kde=True, ax=axes[1], color='#059669')
axes[1].set_title('Load Factor (Mean / Peak)', fontweight='bold')
axes[1].set_xlabel('Load Factor (Stability)')

sns.histplot(corrs, bins=35, kde=True, ax=axes[2], color='#7c3aed')
axes[2].axvline(corrs.median(), color='#dc2626', linestyle='--', label=f'Median = {corrs.median():.2f}')
axes[2].set_title('Correlation with Aggregate Grid', fontweight='bold')
axes[2].set_xlabel('Pearson r')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Consumer Archetypes (Load Signatures)

In [ ]:
# Exemplary meters representing 4 key consumer categories
archetypes = [
    ('Commercial / Office (9-5 Peak, Weekend Collapse)', 'MT_333', '#2563eb'),
    ('Residential (Evening Peak, Active Weekends)', 'MT_222', '#ea580c'),
    ('24/7 Continuous Industrial Plant', 'MT_081', '#059669'),
    ('Peaky / Intermittent Consumer', 'MT_019', '#9333ea')
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
axes = axes.flatten()

for idx, (label, meter_id, color) in enumerate(archetypes):
    m_df = pd.DataFrame({'load': df_stable[meter_id]})
    m_df['hour'] = m_df.index.hour + m_df.index.minute / 60.0
    m_df['is_weekend'] = m_df.index.dayofweek.isin([5, 6])
    
    wk = m_df[~m_df['is_weekend']].groupby('hour')['load'].mean()
    we = m_df[m_df['is_weekend']].groupby('hour')['load'].mean()
    
    axes[idx].plot(wk.index, wk.values, color=color, linewidth=2.2, label='Weekday')
    axes[idx].plot(we.index, we.values, color='#64748b', linestyle='--', linewidth=2, label='Weekend')
    axes[idx].set_title(f"{label} ({meter_id})", fontweight='bold')
    axes[idx].set_ylabel('Power (kW)')
    axes[idx].legend(loc='upper left')
    if idx >= 2:
        axes[idx].set_xlabel('Hour of Day')
        axes[idx].set_xticks(range(0, 25, 4))

plt.tight_layout()
plt.show()

## 5. Autocorrelation (ACF / PACF) for Feature Engineering

In [ ]:
n_lags = 1344 # 14 days at 15-minute resolution
acf_vals = acf(agg_stable.dropna(), nlags=n_lags, fft=True)
pacf_vals = pacf(agg_stable.dropna(), nlags=96, method='ywm')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
lags_hours = np.arange(len(acf_vals)) * 15.0 / 60.0
ax1.plot(lags_hours, acf_vals, color='#2563eb', linewidth=1.5)
ax1.axvline(24, color='#ea580c', linestyle='--', label='24h Daily Cycle (r=0.93)')
ax1.axvline(168, color='#dc2626', linestyle='--', label='7d Weekly Cycle (r=0.89)')
ax1.set_title('Autocorrelation (ACF) up to 14 Days', fontweight='bold')
ax1.set_xlabel('Lag (Hours)')
ax1.set_ylabel('Autocorrelation (r)')
ax1.legend(loc='upper right')

pacf_lags = np.arange(len(pacf_vals)) * 15.0 / 60.0
ax2.bar(pacf_lags, pacf_vals, width=0.2, color='#059669')
ax2.axvline(24, color='#dc2626', linestyle='--', label='24h Spike')
ax2.set_title('Partial Autocorrelation (PACF) up to 24 Hours', fontweight='bold')
ax2.set_xlabel('Lag (Hours)')
ax2.set_ylabel('Partial Correlation')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()